# 07 — Repair the ground-truth answers

**CPU, minutes. Needs internet (re-downloads the MATH dataset).**

`answer_gt` in the traces was extracted in notebook 01 with a regex that stops at the first closing brace, so 71 of the 300 problems have truncated reference answers (e.g. `\dfrac{1}{3}` stored as `\dfrac{1`). This re-derives every reference answer from `EleutherAI/hendrycks_math` with brace counting, joining on normalised problem text, and reports how many `correct` flags change.

Writes `hand_labeling_export/repaired_ground_truth.json` (one entry per `model|problem_id`, all traces) and `correctness_audit.json`. The necessity re-run (`09_necessity_rerun.py`) scores against it.

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# =============================================================================
# GROUND-TRUTH REPAIR AND CORRECTNESS AUDIT
#
# THE PROBLEM
# answer_gt was written in notebook 01 using the old parser, whose boxed regex
# stops at the first closing brace. Any answer containing braces was stored
# truncated: \dfrac{1}{3} became "\dfrac{1". This affects the SAME 71 of 300
# problems in every model -- it is a property of the problems, not the models.
#
# The `correct` flag was then computed by comparing a truncated prediction
# against a truncated target. That can agree by accident (both cut to the same
# prefix) even when the underlying answers differ, e.g. \dfrac{1}{3} and
# \dfrac{1}{2} both reduce to "\dfrac{1". So `correct` is unreliable for those
# 71 problems -- and `correct` is the eligibility filter used by every
# necessity event-selection script.
#
# The trace files contain no independent copy of the dataset answer (verified:
# 71/71 have no clean gt field), so the only sound source is the dataset
# itself.
#
# WHAT THIS DOES
#   1. Loads EleutherAI/hendrycks_math and indexes canonical answers by
#      problem text, extracting with brace-counting.
#   2. Joins to the local traces on normalised problem text.
#   3. Recomputes correctness properly and reports how many flags CHANGE.
#   4. Writes a repaired ground-truth map. It does NOT modify the traces --
#      the audit comes first.
#
# READ:
#   Few flags change -> existing event selection stands; fix is forward-looking.
#   Many change      -> event pools were selected on bad flags and downstream
#                       necessity results need revisiting.
#
# COST: CPU + internet for the dataset. Minutes.
# =============================================================================



from pathlib import Path
from collections import Counter, defaultdict
import json
import re

GDRIVE_BASE = DATA_ROOT

In [ ]:
TRACES_DIR = GDRIVE_BASE / "traces"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
REPAIRED_OUT = EXPORT_DIR / "repaired_ground_truth.json"
AUDIT_OUT = EXPORT_DIR / "correctness_audit.json"

HENDRYCKS_CONFIGS = [
    "algebra", "counting_and_probability", "geometry", "intermediate_algebra",
    "number_theory", "prealgebra", "precalculus",
]

In [ ]:
def boxed_spans(text):
    r"""Contents of every \boxed{...}, via brace counting so nesting survives."""
    spans = []
    for m in re.finditer(r'\\boxed\s*\{', text or ""):
        start = m.end() - 1
        depth = 0
        for j in range(start, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    spans.append(text[start + 1:j])
                    break
    return spans


def normalise_answer(s):
    r"""Conservative canonicalisation -- must never merge distinct answers."""
    if s is None:
        return "UNKNOWN"
    s = s.strip()
    s = re.sub(r'^\$+|\$+$', '', s)
    s = re.sub(r'\\(?:left|right|,|;|!|quad|qquad)', '', s)
    s = re.sub(r'\\text\s*\{([^}]*)\}', r'\1', s)
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac')
    s = re.sub(r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}', r'\1/\2', s)
    s = re.sub(r'\s+', '', s)
    s = s.rstrip('.,;:')
    s = re.sub(r'^\{|\}$', '', s)
    return s or "UNKNOWN"


def normalise_problem(p):
    """Join key. Whitespace and case are unreliable across sources; the
    mathematical content is not."""
    return re.sub(r'\s+', ' ', (p or "").strip()).lower()


def looks_truncated(s):
    return isinstance(s, str) and s.count('{') != s.count('}')


def build_dataset_index():
    from datasets import load_dataset
    index = {}
    for cfg in HENDRYCKS_CONFIGS:
        print(f"  loading {cfg} ...")
        for split in ("train", "test"):
            try:
                ds = load_dataset("EleutherAI/hendrycks_math", cfg, split=split,
                                  trust_remote_code=False)
            except Exception as e:
                print(f"    {split}: skipped ({e})")
                continue
            for row in ds:
                sol = row.get("solution", "")
                b = boxed_spans(sol)
                if not b:
                    continue
                index[normalise_problem(row.get("problem", ""))] = {
                    "answer_raw": b[-1],
                    "answer_norm": normalise_answer(b[-1]),
                    "level": row.get("level"), "subject": row.get("type") or cfg,
                }
    print(f"  indexed {len(index)} problems from the dataset")
    return index

In [ ]:
def main():
    print("\n" + "#" * 78)
    print("# GROUND-TRUTH REPAIR AND CORRECTNESS AUDIT")
    print("#" * 78)

    print("\nBuilding dataset index...")
    index = build_dataset_index()

    repaired = {}
    audit = defaultdict(Counter)
    examples = []

    for model_dir in sorted(TRACES_DIR.iterdir()):
        if not model_dir.is_dir():
            continue
        model = model_dir.name
        for f in sorted(model_dir.glob("*.json")):
            try:
                o = json.load(open(f))
            except Exception:
                continue
            pid = o.get("problem_id")
            if pid is None or "problem" not in o:
                continue

            key = normalise_problem(o["problem"])
            hit = index.get(key)
            a = audit[model]
            a["total"] += 1

            if hit is None:
                a["unmatched"] += 1
                continue
            a["matched"] += 1

            stored_gt = o.get("answer_gt", "")
            true_gt = hit["answer_norm"]
            if looks_truncated(stored_gt):
                a["gt_was_truncated"] += 1

            # the model's own answer, properly re-extracted
            pred = boxed_spans(o.get("full_response", ""))
            pred_norm = normalise_answer(pred[-1]) if pred else "UNKNOWN"

            old_flag = bool(o.get("correct"))
            new_flag = (pred_norm != "UNKNOWN") and (pred_norm == true_gt)

            if old_flag == new_flag:
                a["flag_unchanged"] += 1
            else:
                a["flag_CHANGED"] += 1
                a["false_positive" if old_flag else "false_negative"] += 1
                if len(examples) < 12:
                    examples.append({
                        "model": model, "problem_id": pid,
                        "stored_gt": stored_gt, "true_gt": true_gt,
                        "pred": pred_norm, "old": old_flag, "new": new_flag,
                    })

            repaired[f"{model}|{pid}"] = {
                "stored_answer_gt": stored_gt, "true_answer_gt": true_gt,
                "pred_normalised": pred_norm,
                "correct_old": old_flag, "correct_new": new_flag,
            }

    print("\n" + "=" * 78)
    print("PER-MODEL AUDIT")
    print("=" * 78)
    hdr = f"  {'Model':<32}{'matched':>8}{'unmatch':>8}{'trunc gt':>9}{'CHANGED':>9}{'FP':>5}{'FN':>5}"
    print(hdr)
    total_changed = 0
    for model, a in sorted(audit.items()):
        total_changed += a["flag_CHANGED"]
        print(f"  {model[-31:]:<32}{a['matched']:>8}{a['unmatched']:>8}"
              f"{a['gt_was_truncated']:>9}{a['flag_CHANGED']:>9}"
              f"{a['false_positive']:>5}{a['false_negative']:>5}")

    if examples:
        print("\n  examples of changed verdicts:")
        for e in examples:
            print(f"    {e['model'][-14:]:<15} {e['problem_id']:<12} "
                  f"stored={e['stored_gt']!r:<16} true={e['true_gt']!r:<16} "
                  f"pred={e['pred']!r:<16} {e['old']} -> {e['new']}")

    print("\n" + "#" * 78)
    print("# READ")
    print("#" * 78)
    any_unmatched = sum(a["unmatched"] for a in audit.values())
    if any_unmatched:
        print(f"  {any_unmatched} problems did not match the dataset by text. If this is")
        print("  more than a handful, the join key needs work before trusting the audit.")
    if total_changed == 0:
        print("  NO correctness flags change. The truncation was cosmetic: predictions and")
        print("  targets were cut identically and the verdicts happened to be right.")
        print("  Existing event selection stands. Fix the parser going forward and note")
        print("  the issue in limitations; no re-run needed.")
    else:
        print(f"  {total_changed} correctness flags CHANGE across all models.")
        print("  `correct` gates eligibility in every necessity event-selection script, so")
        print("  the event pools were built on some wrong flags. Assess whether the")
        print("  affected problems actually entered the pools before deciding on a re-run --")
        print("  a changed flag only matters if that problem was selected.")
        print("  False positives (was 'correct', actually wrong) are the damaging direction:")
        print("  those problems should never have been eligible.")

    json.dump(repaired, open(REPAIRED_OUT, "w"), indent=2)
    json.dump({m: dict(a) for m, a in audit.items()}, open(AUDIT_OUT, "w"), indent=2)
    print(f"\n  Wrote {REPAIRED_OUT}")
    print(f"  Wrote {AUDIT_OUT}")
    print("\n  NOTE: traces were NOT modified. This is an audit; repair the pipeline")
    print("  only once the scale of the problem is understood.")


if __name__ == "__main__":
    main()